# AccentMark: Step 2 - Modeling, Visualization, and Metrics

This notebook contains the core modeling and analysis of the **AccentMark** project.
We load the phoneme-level feature means, align the speakers on common phonemes, compute acoustic deviation vectors against the native English baseline, project these continuous accent fingerprints using PCA and UMAP, and compute quantitative metrics of the embedding space.

## 1. Imports and Configurations

In [ ]:
import os
import pickle
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics.pairwise import cosine_similarity

# Add src to pathway
sys.path.append(os.path.abspath(os.path.join('..')))
from src.fingerprint import build_fingerprint_matrix

# Set style for visualizations
sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Helvetica", "Arial", "DejaVu Sans"]

## 2. Load Extracted Features

In [ ]:
RESULTS_DIR = os.path.join('..', 'results')
pkl_path = os.path.join(RESULTS_DIR, 'all_means.pkl')

with open(pkl_path, 'rb') as f:
    all_means = pickle.load(f)

print(f"Loaded {len(all_means)} speaker feature configurations.")

## 3. Construct Native English Baseline Reference

We build the baseline reference phoneme means by averaging the phoneme mean vectors of the two native speakers `NJS` and `TXHC`.

In [ ]:
SPEAKERS = {
    'ABA': 'Arabic', 'YBAA': 'Arabic',
    'SVBI': 'Hindi', 'TNI': 'Hindi',
    'HJK': 'Korean', 'YDCK': 'Korean',
    'BWC': 'Mandarin', 'LXC': 'Mandarin',
    'EBVS': 'Spanish', 'ERMS': 'Spanish',
    'HQTV': 'Vietnamese', 'PNV': 'Vietnamese',
    'NJS': 'Native', 'TXHC': 'Native'
}

njs_means = all_means.get('NJS', {})
txhc_means = all_means.get('TXHC', {})

# Merge all phonemes found in either reference speaker
ref_phonemes = set(njs_means.keys()).union(txhc_means.keys())
reference_means = {}

for ph in ref_phonemes:
    if ph in njs_means and ph in txhc_means:
        reference_means[ph] = (njs_means[ph] + txhc_means[ph]) / 2.0
    elif ph in njs_means:
        reference_means[ph] = njs_means[ph]
    else:
        reference_means[ph] = txhc_means[ph]

print(f"Reference constructed with {len(reference_means)} phonemes.")

## 4. Build Fingerprint Matrix for Non-Native Speakers

We align non-native speakers on common phonemes and compute their concatenated deviation vectors.

In [ ]:
non_native_speakers = [s for s, l1 in SPEAKERS.items() if l1 != 'Native']

# Construct fingerprint matrix
fingerprint_matrix, common_phonemes = build_fingerprint_matrix(
    all_speaker_means=all_means, 
    reference_means=reference_means, 
    speaker_ids=non_native_speakers
)

n_speakers, fingerprint_dim = fingerprint_matrix.shape
print(f"Common Phonemes across all speakers ({len(common_phonemes)}): {common_phonemes}")
print(f"Fingerprint Matrix Shape: {fingerprint_matrix.shape} (Speakers: {n_speakers}, Dim: {fingerprint_dim})")

## 5. Dimensionality Reduction (PCA & UMAP)

We scale the fingerprints, perform PCA to project the components, and then run UMAP for non-linear manifold rendering.

In [ ]:
# Scale features
scaler = StandardScaler()
scaled_fingerprints = scaler.fit_transform(fingerprint_matrix)

# PCA projection
n_components = min(10, n_speakers - 1)
pca = PCA(n_components=n_components, random_state=42)
pca_embeddings = pca.fit_transform(scaled_fingerprints)
pca_variance = float(np.sum(pca.explained_variance_ratio_))

print(f"PCA Variance Explained (top {n_components} components): {pca_variance * 100:.2f}%")

# UMAP Projection
umap_model = umap.UMAP(n_components=2, n_neighbors=5, random_state=42)
umap_embeddings = umap_model.fit_transform(scaled_fingerprints)

# Compute Silhouette Score on UMAP embedding with L1 labels
l1_labels = [SPEAKERS[spk] for spk in non_native_speakers]
sil_score = float(silhouette_score(umap_embeddings, l1_labels))
print(f"UMAP Silhouette Score (L1 Language labels): {sil_score:.4f}")

## 6. Generate Visualizations

### Visual 1: UMAP Clusters
Scatter plot of speakers in the UMAP space, colored by their native L1 background.

In [ ]:
plt.figure(figsize=(10, 7), dpi=150)
unique_l1s = sorted(list(set(l1_labels)))
colors = sns.color_palette("Set2", len(unique_l1s))
l1_to_color = dict(zip(unique_l1s, colors))

for l1 in unique_l1s:
    indices = [i for i, label in enumerate(l1_labels) if label == l1]
    plt.scatter(
        umap_embeddings[indices, 0], 
        umap_embeddings[indices, 1], 
        label=l1, 
        color=l1_to_color[l1], 
        s=150, 
        edgecolors='black', 
        alpha=0.85
    )

# Annotate speaker IDs
for i, spk in enumerate(non_native_speakers):
    plt.annotate(
        spk, 
        (umap_embeddings[i, 0], umap_embeddings[i, 1]),
        textcoords="offset points",
        xytext=(0, 10),
        ha='center',
        fontsize=9,
        weight='bold'
    )

plt.title(f"AccentMark UMAP Projection (Silhouette Score: {sil_score:.3f})", fontsize=14, weight='bold', pad=15)
plt.xlabel("UMAP Dimension 1", fontsize=12)
plt.ylabel("UMAP Dimension 2", fontsize=12)
plt.legend(title="L1 Language Background", loc='best', frameon=True)
plt.tight_layout()

umap_plot_path = os.path.join(RESULTS_DIR, 'umap_clusters.png')
plt.savefig(umap_plot_path, bbox_inches='tight')
plt.close()
print(f"Saved UMAP plot to {umap_plot_path}")

### Visual 2: Phoneme Deviation Heatmap
Heatmap showing the magnitude (L2 norm) of the pronunciation deviation vector for each phoneme across all speakers.

In [ ]:
# Compute phoneme-level deviation norms (L2 norm of length 17 deviation vector)
heatmap_data = []
for spk in non_native_speakers:
    row = []
    for ph in common_phonemes:
        dev = all_means[spk][ph] - reference_means[ph]
        row.append(np.linalg.norm(dev))
    heatmap_data.append(row)

df_heatmap = pd.DataFrame(
    heatmap_data, 
    index=[f"{spk} ({SPEAKERS[spk]})" for spk in non_native_speakers],
    columns=common_phonemes
)

plt.figure(figsize=(20, 8), dpi=150)
sns.heatmap(
    df_heatmap, 
    annot=True, 
    fmt=".2f", 
    cmap="YlOrRd", 
    cbar_kws={'label': 'Pronunciation Deviation Magnitude (L2 Norm)'},
    linewidths=0.5
)
plt.title("AccentMark Phoneme-Level Accent Fingerprint (Deviation Magnitude against Native Reference)", fontsize=16, weight='bold', pad=20)
plt.ylabel("Speaker (L1 Background)", fontsize=13)
plt.xlabel("Common Phonemes", fontsize=13)
plt.xticks(rotation=0, fontsize=11, weight='semibold')
plt.yticks(rotation=0, fontsize=11)
plt.tight_layout()

heatmap_plot_path = os.path.join(RESULTS_DIR, 'phoneme_heatmap.png')
plt.savefig(heatmap_plot_path, bbox_inches='tight')
plt.close()
print(f"Saved heatmap to {heatmap_plot_path}")

### Visual 3: Fingerprint Comparison for 3 Different L1 Speakers
Side-by-side bar charts showing deviation magnitude across common phonemes.

In [ ]:
# Pick 3 speakers from different backgrounds
comp_speakers = ['ABA', 'SVBI', 'HJK'] # Arabic, Hindi, Korean

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True, dpi=150)
palette = sns.color_palette("husl", 3)

for idx, spk in enumerate(comp_speakers):
    ax = axes[idx]
    l1 = SPEAKERS[spk]
    
    # Get deviation magnitudes for this speaker
    dev_mags = df_heatmap.loc[f"{spk} ({l1})"]
    
    sns.barplot(
        x=common_phonemes, 
        y=dev_mags.values, 
        ax=ax, 
        color=palette[idx], 
        edgecolor="black", 
        alpha=0.85
    )
    ax.set_title(f"{spk} (L1: {l1})", fontsize=13, weight='bold')
    ax.set_xlabel("Phoneme", fontsize=11)
    if idx == 0:
        ax.set_ylabel("Deviation Magnitude (L2 Norm)", fontsize=11)
    ax.set_ylim(0, max(df_heatmap.values.max() * 1.15, 1.0))
    for p in ax.patches:
        height = p.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(p.get_x() + p.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=8)

plt.suptitle("Acoustic Fingerprint Comparison Across Different L1 backgrounds", fontsize=15, weight='bold', y=1.02)
plt.tight_layout()

comparison_plot_path = os.path.join(RESULTS_DIR, 'fingerprint_comparison.png')
plt.savefig(comparison_plot_path, bbox_inches='tight')
plt.close()
print(f"Saved fingerprint comparison plot to {comparison_plot_path}")

## 7. Speaker Re-Identification & L1 Cosine Distance Ratio

We use split-half fingerprints to test stability. 
For speaker verification, we perform Leave-One-Out validation on PCA embeddings using KNN (k=1) to see if each split correctly matches the other split of the same speaker.

In [ ]:
# Construct split speakers list and true labels
split_speakers = []
split_speaker_labels = []
split_l1_labels = []

for spk in non_native_speakers:
    split_speakers.append(f"{spk}_split1")
    split_speaker_labels.append(spk)
    split_l1_labels.append(SPEAKERS[spk])
    
    split_speakers.append(f"{spk}_split2")
    split_speaker_labels.append(spk)
    split_l1_labels.append(SPEAKERS[spk])

# Build fingerprint matrix for the split-half segments
split_fingerprints, split_common_phonemes = build_fingerprint_matrix(
    all_speaker_means=all_means,
    reference_means=reference_means,
    speaker_ids=split_speakers
)

# Scale and PCA project the split-half fingerprints
split_scaler = StandardScaler()
split_scaled = split_scaler.fit_transform(split_fingerprints)

split_pca = PCA(n_components=min(10, len(split_speakers) - 1), random_state=42)
split_pca_embeddings = split_pca.fit_transform(split_scaled)

# Leave-One-Out KNN (k=1) speaker re-identification
correct_reid = 0
total_reid = len(split_speakers)

for i in range(total_reid):
    # Split train and test indices
    train_indices = [idx for idx in range(total_reid) if idx != i]
    X_train = split_pca_embeddings[train_indices]
    y_train = [split_speaker_labels[idx] for idx in train_indices]
    
    X_test = split_pca_embeddings[i].reshape(1, -1)
    y_test = split_speaker_labels[i]
    
    knn = KNeighborsClassifier(n_neighbors=1, metric='euclidean')
    knn.fit(X_train, y_train)
    pred = knn.predict(X_test)[0]
    
    if pred == y_test:
        correct_reid += 1
        
reid_accuracy = float(correct_reid / total_reid)
print(f"Speaker Re-Identification Accuracy (LOO KNN k=1): {reid_accuracy * 100:.2f}%")

### Calculate Same-L1 vs Different-L1 Cosine Distance Ratio
Using the full session fingerprints of the 12 non-native speakers, we compute pairwise cosine distances and analyze within-L1 vs cross-L1 distance relationships.

In [ ]:
# Pairwise similarities and distances
similarities = cosine_similarity(fingerprint_matrix)
cosine_dists = 1.0 - similarities

same_l1_dists = []
diff_l1_dists = []

for i in range(n_speakers):
    for j in range(i + 1, n_speakers):
        spk1, spk2 = non_native_speakers[i], non_native_speakers[j]
        l1_1, l1_2 = SPEAKERS[spk1], SPEAKERS[spk2]
        
        d = float(cosine_dists[i, j])
        if l1_1 == l1_2:
            same_l1_dists.append(d)
        else:
            diff_l1_dists.append(d)

same_l1_avg = float(np.mean(same_l1_dists))
diff_l1_avg = float(np.mean(diff_l1_dists))
distance_ratio = float(diff_l1_avg / same_l1_avg) if same_l1_avg > 0 else 0.0

print(f"Average Cosine Distance (Same L1): {same_l1_avg:.4f}")
print(f"Average Cosine Distance (Different L1): {diff_l1_avg:.4f}")
print(f"Distance Ratio (Diff L1 / Same L1): {distance_ratio:.2f}x")

## 8. Export Metrics to JSON

In [ ]:
metrics = {
    "silhouette_score": sil_score,
    "pca_variance_explained": pca_variance,
    "speaker_reid_accuracy": reid_accuracy,
    "same_l1_avg_distance": same_l1_avg,
    "diff_l1_avg_distance": diff_l1_avg,
    "distance_ratio": distance_ratio,
    "n_common_phonemes": int(len(common_phonemes)),
    "fingerprint_dim": int(fingerprint_dim)
}

metrics_json_path = os.path.join(RESULTS_DIR, 'metrics.json')
with open(metrics_json_path, 'w') as f:
    json.dump(metrics, f, indent=4)

print(f"Saved metrics JSON to {metrics_json_path}:")
print(json.dumps(metrics, indent=2))